
# AI-Based Job & Skill Recommendation - Colab Notebook

This notebook is designed for **Google Colab**.

## What it does
1. Loads `job_recommendation_dataset.csv`
2. Performs basic EDA
3. Builds a **baseline content-based recommender** using **TF-IDF + cosine similarity**
4. Adds **skill-gap analysis**
5. Shows an **optional advanced semantic model** using Sentence Transformers
6. Provides simple evaluation templates

> Recommended for your current dataset because the dataset contains **job records**, not user-job interaction labels.


In [5]:

# =========================
# 1. Install dependencies
# =========================
!pip -q install sentence-transformers scikit-learn pandas numpy


In [6]:

# =========================
# 2. Imports
# =========================
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



## Upload dataset in Colab
- Click the folder icon
- Upload `job_recommendation_dataset.csv`
- Or mount Google Drive and update the path


In [8]:

# =========================
# 2. Load dataset from Drive
# =========================
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Machine Learning/DataSet/job_recommendation_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
display(df.head())

Shape: (50000, 7)

Columns: ['Job Title', 'Company', 'Location', 'Experience Level', 'Salary', 'Industry', 'Required Skills']


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills
0,Early years teacher,Richardson Ltd,Sydney,Senior Level,87000.0,Healthcare,Pharmaceuticals
1,Counselling psychologist,"Ramos, Santiago and Stewart",San Francisco,Mid Level,50000.0,Marketing,"Google Ads, SEO, Content Writing"
2,Radio broadcast assistant,Franco Group,New York,Mid Level,77000.0,Healthcare,"Patient Care, Nursing, Medical Research, Pharm..."
3,"Designer, exhibition/display",Collins Inc,Berlin,Senior Level,90000.0,Software,Machine Learning
4,"Psychotherapist, dance movement",Barker Group,Sydney,Entry Level,112000.0,Healthcare,"Nursing, Medical Research, Pharmaceuticals"


In [9]:

# =========================
# 4. Quick EDA
# =========================
print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

for col in ["Location", "Experience Level", "Industry"]:
    print(f"\nTop values in {col}:")
    display(df[col].value_counts().head(10))

print("\nSalary summary:")
display(df["Salary"].describe())



Missing values:


,0
Job Title,0
Company,0
Location,0
Experience Level,0
Salary,0
Industry,0
Required Skills,0



Duplicate rows: 0

Top values in Location:


,count
Location,
Toronto,7229
London,7223
New York,7167
Sydney,7161
San Francisco,7120
Bangalore,7052
Berlin,7048



Top values in Experience Level:


,count
Experience Level,
Mid Level,16739
Senior Level,16658
Entry Level,16603



Top values in Industry:


,count
Industry,
Software,7302
Manufacturing,7169
Marketing,7158
Education,7144
Retail,7106
Healthcare,7104
Finance,7017



Salary summary:


,Salary
count,50000.000000
mean,95145.100000
std,31782.635648
min,40000.000000
25%,68000.000000
50%,95000.000000
75%,123000.000000
max,150000.000000



## Preprocessing
We create a text field combining:
- Job Title
- Industry
- Experience Level
- Required Skills
- Location

This works well for a **content-based recommendation** setup.


In [10]:

# =========================
# 5. Text preprocessing
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9,+# ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_cols = ["Job Title", "Industry", "Experience Level", "Required Skills", "Location"]

for col in text_cols:
    df[col] = df[col].astype(str).fillna("")

df["combined_text"] = (
    df["Job Title"] + " " +
    df["Industry"] + " " +
    df["Experience Level"] + " " +
    df["Required Skills"] + " " +
    df["Location"]
).apply(clean_text)

display(df[["Job Title", "Required Skills", "combined_text"]].head())


,Job Title,Required Skills,combined_text
0,Early years teacher,Pharmaceuticals,early years teacher healthcare senior level ph...
1,Counselling psychologist,"Google Ads, SEO, Content Writing",counselling psychologist marketing mid level g...
2,Radio broadcast assistant,"Patient Care, Nursing, Medical Research, Pharm...",radio broadcast assistant healthcare mid level...
3,"Designer, exhibition/display",Machine Learning,"designer, exhibition display software senior l..."
4,"Psychotherapist, dance movement","Nursing, Medical Research, Pharmaceuticals","psychotherapist, dance movement healthcare ent..."



## Baseline model: TF-IDF + cosine similarity

This is the best first model for your uploaded dataset because:
- It is **simple**
- It is **fast**
- It is **interpretable**
- It does **not require labels**


In [11]:

# =========================
# 6. Vectorization
# =========================
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
job_matrix = vectorizer.fit_transform(df["combined_text"])

print("TF-IDF matrix shape:", job_matrix.shape)


TF-IDF matrix shape: (50000, 3756)


In [12]:

# =========================
# 7. Utility functions
# =========================
def normalize_skill_list(skills_text):
    skills = [s.strip().lower() for s in str(skills_text).split(",") if s.strip()]
    return sorted(set(skills))

def build_user_query(target_role="", skills=None, industry="", level="", location=""):
    if skills is None:
        skills = []
    parts = [
        target_role,
        industry,
        level,
        location,
        ", ".join(skills)
    ]
    query = " ".join([str(p) for p in parts if str(p).strip()])
    return clean_text(query)

def jaccard_similarity(user_skills, job_skills):
    a = set([s.strip().lower() for s in user_skills if str(s).strip()])
    b = set(normalize_skill_list(job_skills))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def recommend_jobs(user_query, user_skills, top_n=10, alpha=0.8, beta=0.2):
    user_vec = vectorizer.transform([clean_text(user_query)])
    cos_scores = cosine_similarity(user_vec, job_matrix).flatten()

    jacc_scores = df["Required Skills"].apply(lambda x: jaccard_similarity(user_skills, x)).values
    final_scores = alpha * cos_scores + beta * jacc_scores

    out = df.copy()
    out["cosine_score"] = cos_scores
    out["jaccard_score"] = jacc_scores
    out["final_score"] = final_scores

    out = out.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)
    return out[[
        "Job Title", "Company", "Location", "Experience Level",
        "Salary", "Industry", "Required Skills",
        "cosine_score", "jaccard_score", "final_score"
    ]]

def skill_gap_analysis(user_skills, recommended_jobs_df):
    user_skill_set = set([s.strip().lower() for s in user_skills if str(s).strip()])
    results = []

    for _, row in recommended_jobs_df.iterrows():
        required = set(normalize_skill_list(row["Required Skills"]))
        missing = sorted(required - user_skill_set)
        matched = sorted(required & user_skill_set)

        results.append({
            "Job Title": row["Job Title"],
            "Required Skills": ", ".join(sorted(required)),
            "Matched Skills": ", ".join(matched),
            "Missing Skills": ", ".join(missing),
            "Missing Skill Count": len(missing)
        })

    return pd.DataFrame(results)


In [13]:

# =========================
# 8. Example recommendation
# =========================
candidate_role = "data analyst"
candidate_skills = ["python", "sql", "excel"]
candidate_industry = "software"
candidate_level = "entry level"
candidate_location = "london"

query = build_user_query(
    target_role=candidate_role,
    skills=candidate_skills,
    industry=candidate_industry,
    level=candidate_level,
    location=candidate_location
)

top_jobs = recommend_jobs(query, candidate_skills, top_n=10, alpha=0.8, beta=0.2)
display(top_jobs)


,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,cosine_score,jaccard_score,final_score
0,Intelligence analyst,Keller and Sons,Berlin,Mid Level,56000.0,Software,"Python, SQL",0.438637,0.666667,0.484243
1,Risk analyst,Hampton and Sons,Sydney,Entry Level,125000.0,Software,"SQL, C++, Python",0.473657,0.500000,0.478926
2,Intelligence analyst,Flynn-Lee,New York,Mid Level,50000.0,Finance,"Python, SQL, Excel",0.335124,1.000000,0.468099
3,Cytogeneticist,Wise-Gutierrez,London,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.333999,1.000000,0.467199
4,Systems analyst,Stein-Howard,London,Entry Level,68000.0,Software,"AWS, SQL, Python",0.439753,0.500000,0.451802
5,Proofreader,"Riley, Allen and Buck",Berlin,Entry Level,115000.0,Finance,"Python, SQL, Excel",0.298672,1.000000,0.438937
6,Lawyer,Jones-Moore,Berlin,Entry Level,79000.0,Finance,"Python, SQL, Excel",0.296718,1.000000,0.437375
7,Energy engineer,Jennings-Flowers,Toronto,Entry Level,130000.0,Finance,"Python, SQL, Excel",0.295621,1.000000,0.436497
8,Dietitian,Fields PLC,New York,Entry Level,70000.0,Finance,"Python, SQL, Excel",0.290262,1.000000,0.432209
9,Financial adviser,Suarez-Williamson,Berlin,Entry Level,139000.0,Finance,"Python, SQL, Excel",0.289706,1.000000,0.431765


In [14]:

# =========================
# 9. Skill gap analysis
# =========================
gap_df = skill_gap_analysis(candidate_skills, top_jobs)
display(gap_df)


,Job Title,Required Skills,Matched Skills,Missing Skills,Missing Skill Count
0,Intelligence analyst,"python, sql","python, sql",,0
1,Risk analyst,"c++, python, sql","python, sql",c++,1
2,Intelligence analyst,"excel, python, sql","excel, python, sql",,0
3,Cytogeneticist,"excel, python, sql","excel, python, sql",,0
4,Systems analyst,"aws, python, sql","python, sql",aws,1
5,Proofreader,"excel, python, sql","excel, python, sql",,0
6,Lawyer,"excel, python, sql","excel, python, sql",,0
7,Energy engineer,"excel, python, sql","excel, python, sql",,0
8,Dietitian,"excel, python, sql","excel, python, sql",,0
9,Financial adviser,"excel, python, sql","excel, python, sql",,0



## Optional advanced model: Sentence-BERT

Use this only **after** the TF-IDF baseline is working.
It helps when wording is different but meaning is similar.


In [15]:

# =========================
# 10. Optional semantic embeddings
# =========================
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
job_embeddings = semantic_model.encode(df["combined_text"].tolist(), show_progress_bar=True, convert_to_numpy=True)

def recommend_jobs_sbert(user_query, user_skills, top_n=10, alpha=0.85, beta=0.15):
    user_emb = semantic_model.encode([clean_text(user_query)], convert_to_numpy=True)
    cos_scores = cosine_similarity(user_emb, job_embeddings).flatten()
    jacc_scores = df["Required Skills"].apply(lambda x: jaccard_similarity(user_skills, x)).values
    final_scores = alpha * cos_scores + beta * jacc_scores

    out = df.copy()
    out["semantic_score"] = cos_scores
    out["jaccard_score"] = jacc_scores
    out["final_score"] = final_scores

    out = out.sort_values("final_score", ascending=False).head(top_n).reset_index(drop=True)
    return out[[
        "Job Title", "Company", "Location", "Experience Level",
        "Salary", "Industry", "Required Skills",
        "semantic_score", "jaccard_score", "final_score"
    ]]

top_jobs_semantic = recommend_jobs_sbert(query, candidate_skills, top_n=10)
display(top_jobs_semantic)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

,Job Title,Company,Location,Experience Level,Salary,Industry,Required Skills,semantic_score,jaccard_score,final_score
0,"Engineer, production","Castaneda, Carpenter and Mckinney",London,Entry Level,56000.0,Finance,"Excel, SQL, Python",0.855499,1.00,0.877174
1,"Investment banker, corporate",Hill-Marquez,London,Entry Level,54000.0,Finance,"SQL, Excel, Python",0.847964,1.00,0.870770
2,"Engineer, control and instrumentation","Gonzalez, Wells and Williams",London,Mid Level,131000.0,Finance,"Excel, Python, SQL",0.818105,1.00,0.845389
3,"Secretary, company","Scott, Cortez and Leonard",Toronto,Entry Level,95000.0,Finance,"Excel, SQL, Python",0.795660,1.00,0.826311
4,Software engineer,"Small, Scott and Wilson",London,Entry Level,85000.0,Finance,"Excel, Python, Risk Analysis, SQL",0.836271,0.75,0.823330
5,"Engineer, automotive",Davis-Nguyen,New York,Entry Level,47000.0,Finance,"SQL, Excel, Python",0.791216,1.00,0.822534
6,"Civil engineer, contracting","Stanton, Terry and Rodriguez",Berlin,Entry Level,95000.0,Finance,"SQL, Python, Excel",0.783303,1.00,0.815808
7,Intelligence analyst,Flynn-Lee,New York,Mid Level,50000.0,Finance,"Python, SQL, Excel",0.781456,1.00,0.814237
8,Secretary/administrator,Lopez-Schroeder,London,Entry Level,129000.0,Finance,"Excel, Python, Financial Modeling, SQL",0.822319,0.75,0.811471
9,"Programmer, systems",Berg Inc,London,Entry Level,41000.0,Finance,"SQL, Risk Analysis, Python, Excel",0.821643,0.75,0.810896



## Evaluation ideas for your paper

Because this dataset does **not** include explicit user-job relevance labels, use one of these:

### Option A: Small human evaluation
Create 30-50 test queries manually and judge whether top-5 results are relevant.

Metrics:
- Precision@5
- Recall@5
- MRR
- nDCG@5

### Option B: Proxy evaluation
Treat `Job Title + Required Skills` as pseudo-ground truth:
- query with a subset of skills
- check whether jobs with matching title/industry appear near the top

### Option C: Collect real feedback
If your mobile app collects clicks, saves, or applications, then later train learning-to-rank models.


In [16]:

# =========================
# 11. Simple Precision@K template
# =========================
def precision_at_k(recommended_ids, relevant_ids, k=5):
    rec_k = recommended_ids[:k]
    if k == 0:
        return 0
    return len(set(rec_k) & set(relevant_ids)) / k

# Example placeholder
recommended_example = [1, 5, 7, 9, 11]
relevant_example = [5, 7, 15]

print("Precision@5 =", precision_at_k(recommended_example, relevant_example, k=5))


Precision@5 = 0.4


In [17]:
def calculate_metrics(recommend_func, test_cases, k=5):
    precisions = []
    recalls = []

    for case in test_cases:
        # Get recommendations
        results = recommend_func(case['query'], case['skills'], top_n=k)

        # A result is 'relevant' if it contains at least one of the candidate skills
        # (Proxy relevance for unsupervised data)
        relevant_count = 0
        for _, row in results.iterrows():
            job_skills = set(normalize_skill_list(row['Required Skills']))
            user_skills = set(case['skills'])
            if user_skills.intersection(job_skills):
                relevant_count += 1

        precision = relevant_count / k
        recall = relevant_count / len(case['skills']) if len(case['skills']) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    f1 = 2 * (avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0
    # Accuracy in recommendation is often represented by Hit Rate @ K
    accuracy = np.mean([1 if p > 0 else 0 for p in precisions])

    return accuracy, avg_p, avg_r, f1

# Define 5 diverse test scenarios from the dataset context
eval_queries = [
    {'query': 'data analyst python sql', 'skills': ['python', 'sql', 'excel']},
    {'query': 'software developer java', 'skills': ['java', 'spring', 'hibernate']},
    {'query': 'project manager agile', 'skills': ['agile', 'scrum', 'kanban']},
    {'query': 'cloud architect aws', 'skills': ['aws', 'docker', 'kubernetes']},
    {'query': 'ux designer figma', 'skills': ['figma', 'sketch', 'adobe xd']}
]

print("Evaluating TF-IDF Model...")
acc_t, p_t, r_t, f1_t = calculate_metrics(recommend_jobs, eval_queries)

print("Evaluating S-BERT Model...")
# Note: ensure cell e719125d (SBERT) was defined/run or use a wrapper
acc_s, p_s, r_s, f1_s = calculate_metrics(recommend_jobs_sbert, eval_queries)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy (Hit Rate)', 'Precision@5', 'Recall@5', 'F1-Score'],
    'TF-IDF + Jaccard': [acc_t, p_t, r_t, f1_t],
    'S-BERT + Jaccard': [acc_s, p_s, r_s, f1_s]
})

display(metrics_df.round(4))

Evaluating TF-IDF Model...
Evaluating S-BERT Model...


,Metric,TF-IDF + Jaccard,S-BERT + Jaccard
0,Accuracy (Hit Rate),0.60,0.60
1,Precision@5,0.60,0.60
2,Recall@5,1.00,1.00
3,F1-Score,0.75,0.75



## Final recommendation for your project

### Best fit now
**Primary model:** TF-IDF + cosine similarity + Jaccard skill overlap

### Best stronger version
**Improved model:** Sentence-BERT + Jaccard skill overlap

### Not recommended right now
- **GRU/LSTM**: you do not have time-series user behavior in this CSV
- **Pure classification models**: there is no label like `applied`, `clicked`, `hired`
- **Collaborative filtering**: there is no user-item interaction matrix



## Next step
If you later upload:
- resume dataset
- career guidance dataset
- jobsFE.csv

then we can build a much stronger **multi-input pipeline** and align it with your full research proposal.
